# 🍌 EfficientNet-B0 Training - Banana Leaf Disease Detection

**Production Training Pipeline - Matches train_model.py**

This notebook demonstrates the complete EfficientNet-B0 training pipeline with:
- Pre-generation of augmented dataset (669 → 2,654 images)
- Transfer learning from ImageNet
- Data split: 70% train / 30% validation
- Class balancing (minimum 300 samples per class)

## 🧠 Architecture Overview
- **Model:** EfficientNet-B0 (CNN with MBConv blocks)
- **Base Parameters:** ~4M (frozen, from ImageNet)
- **Custom Head:** ~1.3M trainable parameters
- **Total:** ~5.3M parameters
- **Strategy:** Transfer Learning with frozen base

---

## 1️⃣ Setup and Imports

In [ ]:
# Import required libraries
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path
import shutil
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2️⃣ Configuration

In [ ]:
# Configuration (matches train_model.py)
DATASET_DIR = Path('dataset')
AUGMENTED_DIR = Path('dataset_augmented')
OUTPUT_DIR = Path('training_output')
MODELS_DIR = OUTPUT_DIR / 'models'
GRAPHS_DIR = OUTPUT_DIR / 'graphs'
LOGS_DIR = OUTPUT_DIR / 'logs'

# Create directories
for dir_path in [OUTPUT_DIR, MODELS_DIR, GRAPHS_DIR, LOGS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Model parameters
IMG_HEIGHT = 224  # EfficientNet-B0 standard
IMG_WIDTH = 224
CHANNELS = 3
BATCH_SIZE = 16  # Balanced for memory
EPOCHS = 100     # Max epochs with early stopping
LEARNING_RATE = 0.0005

# Augmentation parameters
MIN_SAMPLES_PER_CLASS = 300  # Target minimum
AUGMENT_THRESHOLD = 300      # Augment if below this

# Data split
TRAIN_SPLIT = 0.70  # 70% training
VAL_SPLIT = 0.15    # 15% validation
TEST_SPLIT = 0.15   # 15% test (combined with val = 30%)

# Class names
CLASS_NAMES = [
    'Black Sigatoka Disease',
    'Bract Mosaic Virus Disease',
    'Cordana Disease',
    'Healthy Leaf',
    'Panama Disease',
    'Pestalotiopsis Disease'
]

print("="*80)
print("✅ Configuration Complete")
print("="*80)
print(f"📂 Original Dataset: {DATASET_DIR}")
print(f"📂 Augmented Dataset: {AUGMENTED_DIR}")
print(f"🏗️  Architecture: EfficientNet-B0 (Transfer Learning)")
print(f"📊 Batch Size: {BATCH_SIZE}")
print(f"🔢 Max Epochs: {EPOCHS}")
print(f"📈 Data Split: {TRAIN_SPLIT*100:.0f}% train / {VAL_SPLIT*100:.0f}% val / {TEST_SPLIT*100:.0f}% test")
print(f"🎯 Augmentation Target: {MIN_SAMPLES_PER_CLASS} images per class")
print(f"💡 Strategy: Pre-augmentation + Real-time augmentation")
print("="*80)

## 3️⃣ Load Original Dataset Information

In [ ]:
# Load dataset info
with open(DATASET_DIR / 'dataset_info.json', 'r') as f:
    dataset_info = json.load(f)

# Display dataset statistics
df_classes = pd.DataFrame(dataset_info['classes'])
df_classes = df_classes.sort_values('count', ascending=False)

print(f"📊 Original Dataset: {dataset_info['dataset_name']}")
print(f"📷 Total Original Images: {dataset_info['total_images']}")
print(f"🏷️  Classes: {dataset_info['num_classes']}\n")

display(df_classes)

# Visualize class distribution
fig, ax = plt.subplots(figsize=(12, 6))
colors = [c['color'] for c in dataset_info['classes']]
bars = ax.bar(df_classes['name'], df_classes['count'], color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.axhline(y=AUGMENT_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Augmentation Threshold ({AUGMENT_THRESHOLD})')
ax.set_title('Original Dataset Class Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Disease Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_ylim([0, max(df_classes['count']) * 1.1])
plt.xticks(rotation=45, ha='right')
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'original_dataset_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n⚠️  Class Imbalance Analysis:")
for _, row in df_classes.iterrows():
    status = "✅ Above threshold" if row['count'] >= AUGMENT_THRESHOLD else "⚠️  Needs augmentation"
    print(f"  {row['name']}: {row['count']} images - {status}")

## 4️⃣ Pre-Generate Augmented Dataset

### 🎯 Two-Stage Augmentation Strategy:
1. **Pre-Augmentation (Heavy):** Generate physical files for minority classes
   - Rotation: ±30°
   - Shifts: ±20%
   - Shear: 20%
   - Zoom: ±20%
   - Flips: Horizontal + Vertical
   - Brightness: 0.7-1.3x

2. **Real-Time Augmentation (Moderate):** During training for additional diversity
   - Applied to all pre-augmented images
   - Lighter transformations

**Result:** 669 original images → ~2,654 augmented images

In [ ]:
print("\n" + "="*80)
print("🔄 PRE-GENERATING AUGMENTED DATASET")
print("="*80)

# Check if already exists
if AUGMENTED_DIR.exists():
    print(f"\n⚠️  Augmented dataset already exists at: {AUGMENTED_DIR}")
    print("\nOptions:")
    print("  1. Use existing (skip augmentation)")
    print("  2. Recreate (delete and regenerate)")
    
    choice = input("\nEnter choice (1 or 2): ").strip()
    
    if choice == '2':
        print("\n🔄 Removing old augmented dataset...")
        shutil.rmtree(AUGMENTED_DIR)
        print("✅ Removed")
    else:
        print("\n✅ Using existing augmented dataset")
        print("="*80)

# Generate if doesn't exist
if not AUGMENTED_DIR.exists():
    print("\n🚀 Starting augmentation process...\n")
    AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)
    
    # Heavy augmentation generator
    augmentation_gen = ImageDataGenerator(
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.7, 1.3],
        fill_mode='nearest'
    )
    
    augmentation_summary = []
    start_time = time.time()
    
    # Process each class
    for idx, class_name in enumerate(CLASS_NAMES, 1):
        print(f"[{idx}/{len(CLASS_NAMES)}] Processing: {class_name}")
        
        class_dir = DATASET_DIR / class_name
        augmented_class_dir = AUGMENTED_DIR / class_name
        augmented_class_dir.mkdir(parents=True, exist_ok=True)
        
        # Get original images
        original_images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.jpeg')) + list(class_dir.glob('*.png'))
        original_count = len(original_images)
        
        print(f"  📷 Original images: {original_count}")
        
        # Copy all originals
        for img_path in original_images:
            shutil.copy2(img_path, augmented_class_dir / img_path.name)
        
        # Augment if needed
        if original_count < AUGMENT_THRESHOLD:
            needed = MIN_SAMPLES_PER_CLASS - original_count
            per_image = (needed // original_count) + 1
            
            print(f"  ⚠️  Below threshold ({AUGMENT_THRESHOLD})")
            print(f"  🎯 Generating {needed} new images ({per_image} per original)")
            
            generated = 0
            for img_path in original_images:
                if generated >= needed:
                    break
                
                # Load and augment
                img = Image.open(img_path)
                img_array = np.array(img)
                img_array = np.expand_dims(img_array, 0)
                
                aug_iter = augmentation_gen.flow(
                    img_array,
                    batch_size=1,
                    save_to_dir=augmented_class_dir,
                    save_prefix=f'aug_{img_path.stem}',
                    save_format='jpg'
                )
                
                for _ in range(per_image):
                    if generated >= needed:
                        break
                    next(aug_iter)
                    generated += 1
            
            final_count = len(list(augmented_class_dir.glob('*.*')))
            print(f"  ✅ Final count: {final_count} images (generated {generated})")
            augmentation_summary.append({
                'class': class_name,
                'original': original_count,
                'augmented': final_count,
                'generated': generated
            })
        else:
            print(f"  ✅ Above threshold - no augmentation needed")
            augmentation_summary.append({
                'class': class_name,
                'original': original_count,
                'augmented': original_count,
                'generated': 0
            })
        print()
    
    elapsed = time.time() - start_time
    
    print("="*80)
    print("✅ AUGMENTATION COMPLETE")
    print("="*80)
    print(f"⏱️  Time taken: {elapsed/60:.1f} minutes\n")
    
    # Summary table
    aug_df = pd.DataFrame(augmentation_summary)
    display(aug_df)
    
    print(f"\n📊 Summary:")
    print(f"  Original total: {aug_df['original'].sum()} images")
    print(f"  Augmented total: {aug_df['augmented'].sum()} images")
    print(f"  Generated: {aug_df['generated'].sum()} new images")
    print(f"  Increase: {(aug_df['augmented'].sum() / aug_df['original'].sum() - 1) * 100:.1f}%")
    print("="*80)
else:
    # Count existing if skipped
    total = 0
    for class_name in CLASS_NAMES:
        count = len(list((AUGMENTED_DIR / class_name).glob('*.*')))
        total += count
    print(f"\n📊 Using existing augmented dataset: {total} total images")
    print("="*80)

## 5️⃣ Create Data Generators

Now create generators from the augmented dataset with **moderate real-time augmentation**.

In [ ]:
print("\n📊 Creating data generators from augmented dataset...\n")

# Training generator with moderate real-time augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,      # Lighter than pre-augmentation
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=VAL_SPLIT + TEST_SPLIT  # 30% for validation
)

# Validation generator (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VAL_SPLIT + TEST_SPLIT
)

# Create generators from AUGMENTED dataset
train_generator = train_datagen.flow_from_directory(
    AUGMENTED_DIR,  # Using augmented dataset
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    AUGMENTED_DIR,  # Using augmented dataset
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print("✅ Data generators created")
print(f"\n📊 Dataset Split:")
print(f"  Training samples: {train_generator.samples} ({TRAIN_SPLIT*100:.0f}%)")
print(f"  Validation samples: {val_generator.samples} ({(VAL_SPLIT+TEST_SPLIT)*100:.0f}%)")
print(f"\n🏷️  Class mapping:")
for class_name, idx in sorted(train_generator.class_indices.items(), key=lambda x: x[1]):
    print(f"  {idx}: {class_name}")

## 6️⃣ Visualize Sample Images

In [ ]:
# Get a batch of images
sample_images, sample_labels = next(train_generator)

# Get class names
class_indices = train_generator.class_indices
class_names_map = {v: k for k, v in class_indices.items()}

# Plot samples
fig, axes = plt.subplots(4, 4, figsize=(15, 15))
fig.suptitle('Sample Training Images (with Real-Time Augmentation)', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(sample_images):
        ax.imshow(sample_images[i])
        label_idx = np.argmax(sample_labels[i])
        ax.set_title(class_names_map[label_idx], fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'sample_training_images.png', dpi=300)
plt.show()

## 7️⃣ Compute Class Weights

In [ ]:
print("⚖️  Computing class weights for balanced training...\n")

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weight_dict = {idx: weight for idx, weight in enumerate(class_weights)}

print("Class Weights:")
for class_name, idx in sorted(train_generator.class_indices.items(), key=lambda x: x[1]):
    print(f"  {class_name}: {class_weight_dict[idx]:.3f}")

# Visualize
plt.figure(figsize=(12, 6))
class_names_ordered = [name for name, _ in sorted(train_generator.class_indices.items(), key=lambda x: x[1])]
plt.bar(class_names_ordered, class_weights, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('Class Weights (Higher = More Emphasis During Training)', fontsize=14, fontweight='bold')
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Weight', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.axhline(y=1.0, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Balanced (1.0)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'class_weights.png', dpi=300)
plt.show()

## 8️⃣ Build EfficientNet-B0 Model

### 🧠 Architecture:
**EfficientNet-B0 is a CNN** with special features:
- **MBConv blocks** (Mobile Inverted Bottleneck Convolutions)
- **Depthwise separable convolutions** (more efficient than standard Conv2D)
- **Squeeze-and-Excitation** blocks for channel attention
- **Compound scaling** (balanced depth, width, resolution)

**Transfer Learning Strategy:**
- Base model: **Frozen** (preserves ImageNet features)
- Custom head: **Trainable** (learns banana disease patterns)

In [ ]:
print("\n🏗️  Building EfficientNet-B0 model...\n")

# Load pre-trained base
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',  # Transfer learning from ImageNet (1.4M images)
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),
    pooling='avg'  # Global average pooling
)

# Freeze base model
base_model.trainable = False

print(f"📊 Base Model: {base_model.name}")
print(f"🔒 Base frozen: {not base_model.trainable}")
print(f"📈 Base parameters: {base_model.count_params():,}\n")

# Build complete model
model = models.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    
    # EfficientNet-B0 base (frozen)
    base_model,
    
    # Custom classification head (trainable)
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Output layer
    layers.Dense(len(CLASS_NAMES), activation='softmax')
], name='EfficientNetB0_BananaDisease')

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2, name='top_2_accuracy')]
)

print("="*80)
model.summary()
print("="*80)

# Parameter breakdown
total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params

print(f"\n📊 Model Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"  Non-trainable (frozen): {non_trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1024 / 1024:.2f} MB")
print(f"\n💡 Why EfficientNet-B0?")
print(f"  ✅ It's a CNN with MBConv blocks (more efficient than standard Conv2D)")
print(f"  ✅ Pre-trained on ImageNet (overcomes limited dataset of 669 images)")
print(f"  ✅ Mobile-optimized (~5M parameters)")
print(f"  ✅ Frozen base prevents overfitting")
print(f"  ✅ Custom head learns banana disease patterns")

## 9️⃣ Setup Training Callbacks

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

callbacks = [
    ModelCheckpoint(
        filepath=str(MODELS_DIR / f'best_model_{timestamp}.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-8,
        verbose=1
    ),
    CSVLogger(
        filename=str(LOGS_DIR / f'training_log_{timestamp}.csv'),
        separator=',',
        append=False
    )
]

print("✅ Callbacks configured:")
print("   1. ModelCheckpoint - Save best model (val_accuracy)")
print("   2. EarlyStopping - Stop if no improvement for 15 epochs")
print("   3. ReduceLROnPlateau - Reduce LR by 0.5x if plateau for 8 epochs")
print("   4. CSVLogger - Log metrics to CSV file")

## 🔟 Train the Model

In [ ]:
print("\n" + "="*80)
print("🚀 STARTING TRAINING")
print("="*80)
print(f"⏱️  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Training samples: {train_generator.samples}")
print(f"📊 Validation samples: {val_generator.samples}")
print(f"🔢 Max epochs: {EPOCHS}")
print(f"📈 Initial LR: {LEARNING_RATE}")
print("="*80 + "\n")

training_start = time.time()

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

training_time = time.time() - training_start

print("\n" + "="*80)
print("✅ TRAINING COMPLETED")
print("="*80)
print(f"⏱️  Training time: {training_time/3600:.2f} hours ({training_time/60:.1f} minutes)")
print(f"📊 Epochs trained: {len(history.history['loss'])}")
print("="*80)

## 1️⃣1️⃣ Plot Training History

In [ ]:
print("\n📈 Generating training analysis graphs...\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('EfficientNet-B0 Training Analysis', fontsize=18, fontweight='bold', y=1.00)

# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train', linewidth=2.5, marker='o', markersize=4)
axes[0, 0].plot(history.history['val_accuracy'], label='Validation', linewidth=2.5, marker='s', markersize=4)
axes[0, 0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].legend(loc='lower right', fontsize=11)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0, 1])

# Loss
axes[0, 1].plot(history.history['loss'], label='Train', linewidth=2.5, marker='o', markersize=4)
axes[0, 1].plot(history.history['val_loss'], label='Validation', linewidth=2.5, marker='s', markersize=4)
axes[0, 1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Loss', fontsize=12)
axes[0, 1].legend(loc='upper right', fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# Top-2 Accuracy
axes[1, 0].plot(history.history['top_2_accuracy'], label='Train Top-2', linewidth=2.5, marker='o', markersize=4)
axes[1, 0].plot(history.history['val_top_2_accuracy'], label='Val Top-2', linewidth=2.5, marker='s', markersize=4)
axes[1, 0].set_title('Top-2 Accuracy', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Accuracy', fontsize=12)
axes[1, 0].legend(loc='lower right', fontsize=11)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1])

# Learning Rate
lr_history = history.history.get('lr', [LEARNING_RATE] * len(history.history['loss']))
axes[1, 1].plot(lr_history, linewidth=2.5, color='orange', marker='o', markersize=4)
axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Learning Rate (log scale)', fontsize=12)
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / f'training_history_{timestamp}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Training graphs saved")

## 1️⃣2️⃣ Evaluate Model & Generate Confusion Matrix

In [ ]:
print("\n🎯 Evaluating model on validation set...\n")

val_generator.reset()
predictions = model.predict(val_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = val_generator.classes

class_names = list(val_generator.class_indices.keys())

# Classification Report
print("\n📊 Classification Report:\n")
report = classification_report(
    true_classes, 
    predicted_classes, 
    target_names=class_names,
    digits=4
)
print(report)

# Save report
with open(LOGS_DIR / f'classification_report_{timestamp}.txt', 'w') as f:
    f.write(report)

# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'},
    linewidths=0.5,
    linecolor='gray'
)
plt.title('Confusion Matrix - EfficientNet-B0', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / f'confusion_matrix_{timestamp}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Confusion matrix saved")

## 1️⃣3️⃣ Per-Class Accuracy Analysis

In [ ]:
# Per-class accuracy
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\n📈 Per-Class Accuracy:")
print("="*50)
accuracy_df = pd.DataFrame({
    'Class': class_names,
    'Accuracy': [f"{acc*100:.2f}%" for acc in class_accuracy],
    'Correct': cm.diagonal(),
    'Total': cm.sum(axis=1)
}).sort_values('Accuracy', ascending=False)

display(accuracy_df)

# Visualize
plt.figure(figsize=(12, 6))
colors_map = {c['name']: c['color'] for c in dataset_info['classes']}
bar_colors = [colors_map.get(name, '#888888') for name in class_names]
plt.bar(class_names, class_accuracy * 100, color=bar_colors, alpha=0.7, edgecolor='black')
plt.axhline(y=np.mean(class_accuracy) * 100, color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(class_accuracy)*100:.2f}%')
plt.title('Per-Class Accuracy - EfficientNet-B0', fontsize=14, fontweight='bold')
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim([0, 100])
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / f'per_class_accuracy_{timestamp}.png', dpi=300)
plt.show()

## 1️⃣4️⃣ Convert to TensorFlow Lite

In [ ]:
print("\n🔄 Converting to TensorFlow Lite...\n")

# Convert
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Get best accuracy for filename
best_val_accuracy = max(history.history['val_accuracy'])
accuracy_pct = f"{best_val_accuracy * 100:.2f}"
model_name = f'efficientnet_b0_{accuracy_pct}'

# Save TFLite
tflite_path = MODELS_DIR / f'{model_name}.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

# Save labels
labels_path = MODELS_DIR / f'labels_{timestamp}.txt'
with open(labels_path, 'w') as f:
    for class_name in CLASS_NAMES:
        f.write(f"{class_name}\n")

# File sizes
keras_size = os.path.getsize(MODELS_DIR / f'best_model_{timestamp}.keras') / (1024 * 1024)
tflite_size = os.path.getsize(tflite_path) / (1024 * 1024)

print(f"✅ Keras model: {keras_size:.2f} MB")
print(f"✅ TFLite model: {tflite_size:.2f} MB")
print(f"✅ Compression: {(1 - tflite_size/keras_size)*100:.1f}%")
print(f"\n📁 Files saved:")
print(f"  TFLite: {tflite_path}")
print(f"  Labels: {labels_path}")

## 1️⃣5️⃣ Final Summary

In [ ]:
# Training summary
summary = {
    'timestamp': timestamp,
    'model': 'EfficientNet-B0 (Transfer Learning)',
    'architecture': {
        'type': 'CNN',
        'specific': 'EfficientNet-B0 with MBConv blocks',
        'base_model': 'EfficientNet-B0 (ImageNet pre-trained)',
        'total_parameters': int(total_params),
        'trainable_parameters': int(trainable_params),
        'non_trainable_parameters': int(non_trainable_params),
        'base_frozen': True
    },
    'dataset': {
        'original_images': 669,
        'augmented_images': 2654,
        'train_samples': train_generator.samples,
        'val_samples': val_generator.samples
    },
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'epochs_trained': len(history.history['loss']),
        'initial_learning_rate': LEARNING_RATE,
        'image_size': f"{IMG_HEIGHT}x{IMG_WIDTH}"
    },
    'training_duration': {
        'seconds': float(training_time),
        'minutes': float(training_time / 60),
        'hours': float(training_time / 3600)
    },
    'final_metrics': {
        'train_accuracy': float(history.history['accuracy'][-1]),
        'val_accuracy': float(history.history['val_accuracy'][-1]),
        'train_loss': float(history.history['loss'][-1]),
        'val_loss': float(history.history['val_loss'][-1]),
        'best_val_accuracy': float(best_val_accuracy),
        'best_epoch': int(np.argmax(history.history['val_accuracy'])) + 1
    },
    'per_class_accuracy': {name: float(acc) for name, acc in zip(class_names, class_accuracy)}
}

# Save summary
with open(LOGS_DIR / f'training_summary_{timestamp}.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# Display final summary
print("\n" + "="*80)
print("🎉 TRAINING PIPELINE COMPLETED SUCCESSFULLY")
print("="*80)

print(f"\n🏆 Model Architecture:")
print(f"  Type: CNN (Convolutional Neural Network)")
print(f"  Specific: EfficientNet-B0 with MBConv blocks")
print(f"  Strategy: Transfer Learning from ImageNet")
print(f"  Total Parameters: {total_params:,}")
print(f"  Trainable: {trainable_params:,} | Frozen: {non_trainable_params:,}")

print(f"\n📊 Final Results:")
print(f"  Model Name: {model_name}")
print(f"  Best Val Accuracy: {best_val_accuracy*100:.2f}%")
print(f"  Best Epoch: {summary['final_metrics']['best_epoch']}")
print(f"  Final Train Accuracy: {summary['final_metrics']['train_accuracy']*100:.2f}%")
print(f"  Final Val Accuracy: {summary['final_metrics']['val_accuracy']*100:.2f}%")

print(f"\n⏱️  Training Duration:")
print(f"  {training_time/3600:.2f} hours ({training_time/60:.1f} minutes)")

print(f"\n📁 Output Files:")
print(f"  Keras Model: {MODELS_DIR / f'best_model_{timestamp}.keras'}")
print(f"  TFLite Model: {tflite_path}")
print(f"  Labels: {labels_path}")
print(f"  Graphs: {GRAPHS_DIR}")
print(f"  Logs: {LOGS_DIR}")

print(f"\n🔄 Next Steps for Flutter Integration:")
print(f"  1. Copy {tflite_path.name} to ../assets/models/")
print(f"  2. Copy {labels_path.name} to ../assets/models/")
print(f"  3. Update Flutter app disease detection service")
print(f"  4. Run 'flutter clean && flutter pub get'")
print(f"  5. Test in the app!")

print(f"\n💡 Interview Talking Points:")
print(f"  ✅ 'EfficientNet-B0 is a CNN with MBConv blocks'")
print(f"  ✅ 'Used transfer learning to overcome limited dataset (669 images)'")
print(f"  ✅ 'Pre-augmented to 2,654 images for class balancing'")
print(f"  ✅ 'Achieved {best_val_accuracy*100:.2f}% validation accuracy'")
print(f"  ✅ 'Mobile-optimized for on-device inference'")

print("\n" + "="*80)
print("✅ All done! Model ready for deployment.")
print("="*80)